# Fake News Type Detection
**Cours : Machine Learning – M1 Data Science**

## 1. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('fake.csv')

print('Taille du dataset :', df.shape)
df.head()

In [ ]:
# Types des colonnes
print(df.dtypes)

In [ ]:
# Valeurs manquantes
print(df.isnull().sum())

In [ ]:
# Distribution de la colonne cible
print(df['type'].value_counts())

df['type'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Nombre d articles par type')
plt.xlabel('Type')
plt.ylabel('Nombre')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('distribution_classes.png', dpi=150)
plt.show()

## 2. Prétraitement du texte

In [ ]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def nettoyer_texte(texte):
    # Si le texte est vide, retourner une chaine vide
    if pd.isna(texte):
        return ''
    # Mettre en minuscules
    texte = texte.lower()
    # Supprimer la ponctuation et les chiffres
    texte = re.sub(r'[^a-z\s]', ' ', texte)
    # Supprimer les stopwords
    mots = texte.split()
    mots = [m for m in mots if m not in stop_words]
    return ' '.join(mots)

# Appliquer le nettoyage sur title et text
df['title_clean'] = df['title'].apply(nettoyer_texte)
df['text_clean']  = df['text'].apply(nettoyer_texte)

# Combiner les deux colonnes
df['texte_final'] = df['title_clean'] + ' ' + df['text_clean']

print('Avant :', df['title'].iloc[0])
print('Apres :', df['texte_final'].iloc[0][:200])

In [ ]:
# Remplir les valeurs manquantes
df['domain_rank'] = df['domain_rank'].fillna(df['domain_rank'].median())
df['spam_score']  = df['spam_score'].fillna(0)

## 3. Feature Engineering — TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encoder la colonne cible (bs, bias, fake... -> 0, 1, 2...)
le = LabelEncoder()
y  = le.fit_transform(df['type'])
print('Classes :', list(le.classes_))

# TF-IDF : transformer le texte en vecteurs numériques
tfidf = TfidfVectorizer(max_features=5000)
X     = tfidf.fit_transform(df['texte_final'])

print('Taille de X :', X.shape)

In [ ]:
# Séparation train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train :', X_train.shape)
print('Test  :', X_test.shape)

## 4. Entraînement des modèles

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Modèle 1 : Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print('Random Forest entraine !')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Modèle 2 : Gradient Boosting
# Note : GradientBoosting ne supporte pas les matrices sparse -> .toarray()
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train.toarray(), y_train)

print('Gradient Boosting entraine !')

## 5. Evaluation des modèles

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

# Random Forest
y_pred_rf = rf.predict(X_test)
print('=== Random Forest ===')
print('Accuracy :', round(accuracy_score(y_test, y_pred_rf), 4))
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
# Gradient Boosting
y_pred_gb = gb.predict(X_test.toarray())
print('=== Gradient Boosting ===')
print('Accuracy :', round(accuracy_score(y_test, y_pred_gb), 4))
print(classification_report(y_test, y_pred_gb, target_names=le.classes_))

In [ ]:
# Matrice de confusion - Random Forest
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=le.classes_, yticklabels=le.classes_,
            cmap='Blues')
plt.title('Matrice de confusion — Random Forest')
plt.xlabel('Predit')
plt.ylabel('Reel')
plt.tight_layout()
plt.savefig('confusion_rf.png', dpi=150)
plt.show()

In [ ]:
# Matrice de confusion - Gradient Boosting
cm2 = confusion_matrix(y_test, y_pred_gb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm2, annot=True, fmt='d',
            xticklabels=le.classes_, yticklabels=le.classes_,
            cmap='Purples')
plt.title('Matrice de confusion — Gradient Boosting')
plt.xlabel('Predit')
plt.ylabel('Reel')
plt.tight_layout()
plt.savefig('confusion_gb.png', dpi=150)
plt.show()

## 6. Sauvegarde du modèle

In [ ]:
import joblib

joblib.dump(rf,    'model_rf.joblib')
joblib.dump(gb,    'model_gb.joblib')
joblib.dump(tfidf, 'tfidf.joblib')
joblib.dump(le,    'label_encoder.joblib')

print('Modeles sauvegardes !')

# Test de rechargement
model_charge = joblib.load('model_rf.joblib')
print('Rechargement OK v')